# Quantization Playground — coding workshop (beginner)

Expected time: **35-45 minutes**, individual or pairs, CPU only.

This notebook stays deliberately small. The point is not to use a quantization library, but to see the arithmetic: scale, round, dequantize, measure the error, and then see why outliers and clipping matter.

On the lesson's Try-it screens these are Exercises 2–5 (Exercise 1 there is Notebook 02d); here they are Exercises 1–4.

**What you will learn:**

- to write absmax quantization yourself: one scale, values rounded to integer codes, and back again
  (dequantizing);
- to measure the error that rounding causes, and how it grows from 8 bits down to 2;
- why asymmetric quantization, with a zero point, suits values that are all 0 or more;
- how one outlier spoils the codes of all the other values, and what clipping trades to fix that.


## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `ExerciseGaps: this cell still has gaps marked ___`.
Later cells use your code, so finish each exercise before moving on.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 4 above).
fill_in_solutions = False  # @param {type:"boolean"}

import io
import re
import tokenize

_solutions = {
    1: "import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nnp.set_printoptions(precision=4, suppress=True)\n\n\ndef absmax_quantize(x, bits=8):\n    x = np.asarray(x, dtype=np.float32)\n    qmax = 2 ** (bits - 1) - 1\n    alpha = float(np.max(np.abs(x)))\n    if alpha == 0:\n        return np.zeros_like(x, dtype=np.int32), 1.0\n    scale = qmax / alpha\n    q = np.round(x * scale)\n    q = np.clip(q, -qmax, qmax).astype(np.int32)\n    return q, scale\n\n\ndef absmax_dequantize(q, scale):\n    return np.asarray(q, dtype=np.float32) / scale\n\nvalues = np.array([5.47, 3.08, -7.59, 0.0, -1.85, -4.57, 10.8, 3.02, -1.92], dtype=np.float32)\nq8, s8 = absmax_quantize(values, bits=8)\ndq8 = absmax_dequantize(q8, s8)\n\npd.DataFrame({\n    \"original\": values,\n    \"int8_code\": q8,\n    \"dequantized\": np.round(dq8, 4),\n    \"error\": np.round(values - dq8, 4),\n})",
    2: "rows = []\nfor bits in [8, 4, 3, 2]:\n    q, s = absmax_quantize(values, bits=bits)\n    dq = absmax_dequantize(q, s)\n    rows.append({\n        \"bits\": bits,\n        \"num_absmax_codes\": 2 ** bits - 1,\n        \"num_codes_used_here\": len(set(q.tolist())),\n        \"scale\": s,\n        \"mean_abs_error\": np.mean(np.abs(values - dq)),\n        \"max_abs_error\": np.max(np.abs(values - dq)),\n    })\n\nbit_sweep = pd.DataFrame(rows)\nbit_sweep",
    3: "def asymmetric_quantize(x, bits=8):\n    x = np.asarray(x, dtype=np.float32)\n    qmin, qmax = 0, 2 ** bits - 1\n    xmin, xmax = float(np.min(x)), float(np.max(x))\n    if xmax == xmin:\n        return np.zeros_like(x, dtype=np.int32), 1.0, 0\n    scale = (xmax - xmin) / (qmax - qmin)\n    zero_point = int(np.round(qmin - xmin / scale))\n    q = np.round(x / scale + zero_point)\n    q = np.clip(q, qmin, qmax).astype(np.int32)\n    return q, scale, zero_point\n\n\ndef asymmetric_dequantize(q, scale, zero_point):\n    return (np.asarray(q, dtype=np.float32) - zero_point) * scale\n\nactivations = np.array([0.0, 0.1, 0.4, 0.9, 1.0, 1.4, 2.0, 2.2, 2.8], dtype=np.float32)\n\nq_sym, s_sym = absmax_quantize(activations, bits=4)\ndq_sym = absmax_dequantize(q_sym, s_sym)\n\nq_asym, s_asym, z_asym = asymmetric_quantize(activations, bits=4)\ndq_asym = asymmetric_dequantize(q_asym, s_asym, z_asym)\n\npd.DataFrame({\n    \"activation\": activations,\n    \"sym_code\": q_sym,\n    \"sym_deq\": np.round(dq_sym, 3),\n    \"asym_code\": q_asym,\n    \"asym_deq\": np.round(dq_asym, 3),\n})",
    4: "def quantize_with_clipping(x, clip_range=5.0, bits=8):\n    clipped = np.clip(x, -clip_range, clip_range)\n    q, s = absmax_quantize(clipped, bits=bits)\n    dq = absmax_dequantize(q, s)\n    return clipped, q, dq, s\n\nsummary = []\nfor clip_range in [None, 10.0, 5.0, 2.5]:\n    if clip_range is None:\n        q, s = absmax_quantize(with_outlier, bits=8)\n        dq = absmax_dequantize(q, s)\n        label = \"no clipping\"\n    else:\n        _, q, dq, s = quantize_with_clipping(with_outlier, clip_range=clip_range, bits=8)\n        label = f\"clip [-{clip_range}, {clip_range}]\"\n    summary.append({\n        \"setting\": label,\n        \"ordinary_codes_used\": len(set(q[:-1].tolist())),\n        \"ordinary_MAE\": np.mean(np.abs(with_outlier[:-1] - dq[:-1])),\n        \"all_values_MAE\": np.mean(np.abs(with_outlier - dq)),\n        \"outlier_dequantized\": dq[-1],\n    })\n\npd.DataFrame(summary)",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


class ExerciseGaps(Exception):
    """A cell still has gaps marked ___."""


def _has_gap(lines):
    """True if `___` appears in the cell as code (not in a string or a comment)."""
    code = "".join(l for l in lines if not l.lstrip().startswith(("%", "!")))
    try:
        return any(t.type == tokenize.NAME and t.string == "___"
                   for t in tokenize.generate_tokens(io.StringIO(code).readline))
    except (tokenize.TokenError, SyntaxError):
        return False


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution.
    A cell that still has gaps ___ stops with a message. (In IPython, ___ is a built-in name, the
    third-to-last output, so without this check the cell would run on with a wrong value.)"""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if m and int(m.group(1)) in _fill_in:
        n = int(m.group(1))
        return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)
    if _has_gap(lines):
        return ['raise ExerciseGaps("this cell still has gaps marked ___: fill them in and run it again, '
                'or tick fill_in_solutions in the first code cell and run that cell")\n']
    return lines


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Part A - absmax quantization from scratch

Implement the symmetric absmax method from the lesson. The range is centered on zero, and the largest absolute value defines the scale.


### ✏️ Exercise 1 — Absmax quantization from scratch

**Goal:** turn floats into integer codes and back, with one scale set by the largest absolute value.

Complete `absmax_quantize` and `absmax_dequantize`.

- `qmax` is the largest positive code of a signed `bits`-bit integer (127 for 8 bits).
- `alpha` is the largest absolute value in `x`.
- The scale maps `alpha` to `qmax`; each value is multiplied by it and rounded.
- Dequantizing divides the codes by the scale.

<details><summary><b>Hints</b> (click to open)</summary>

- A signed `bits`-bit integer has codes from −(2^(bits−1) − 1) to 2^(bits−1) − 1.
- scale = qmax / alpha, so the largest value lands exactly on qmax.
- Quantize: multiply by the scale. Dequantize: divide by it.

</details>

In [ ]:
# Exercise 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


def absmax_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmax = 2 ** (___) - 1
    alpha = float(np.max(np.abs(___)))
    if alpha == 0:
        return np.zeros_like(x, dtype=np.int32), 1.0
    scale = ___ / ___
    q = np.round(___)
    q = np.clip(q, -qmax, qmax).astype(np.int32)
    return q, scale


def absmax_dequantize(q, scale):
    return np.asarray(q, dtype=np.float32) / ___

values = np.array([5.47, 3.08, -7.59, 0.0, -1.85, -4.57, 10.8, 3.02, -1.92], dtype=np.float32)
q8, s8 = absmax_quantize(values, bits=8)
dq8 = absmax_dequantize(q8, s8)

pd.DataFrame({
    "original": values,
    "int8_code": q8,
    "dequantized": np.round(dq8, 4),
    "error": np.round(values - dq8, 4),
})


In [ ]:
# Check for Exercise 1. Run this cell as is.
assert abs(s8 - 127 / 10.8) < 1e-4, f"scale should be 127 / 10.8, got {s8}"
assert q8.tolist() == [64, 36, -89, 0, -22, -54, 127, 36, -23], q8.tolist()
assert np.allclose(dq8, q8 / s8)
_q4, _ = absmax_quantize(values, bits=4)
assert _q4.max() == 7, "4-bit signed codes go up to 7"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


def absmax_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmax = 2 ** (bits - 1) - 1
    alpha = float(np.max(np.abs(x)))
    if alpha == 0:
        return np.zeros_like(x, dtype=np.int32), 1.0
    scale = qmax / alpha
    q = np.round(x * scale)
    q = np.clip(q, -qmax, qmax).astype(np.int32)
    return q, scale


def absmax_dequantize(q, scale):
    return np.asarray(q, dtype=np.float32) / scale

values = np.array([5.47, 3.08, -7.59, 0.0, -1.85, -4.57, 10.8, 3.02, -1.92], dtype=np.float32)
q8, s8 = absmax_quantize(values, bits=8)
dq8 = absmax_dequantize(q8, s8)

pd.DataFrame({
    "original": values,
    "int8_code": q8,
    "dequantized": np.round(dq8, 4),
    "error": np.round(values - dq8, 4),
})
```

</details>

In [ ]:
print("scale:", round(s8, 4))
print("original:    ", values)
print("quantized:   ", q8)
print("dequantized: ", np.round(dq8, 4))
print("mean absolute error:", np.mean(np.abs(values - dq8)))

for v, q, d in zip(values, q8, dq8):
    if q == 36:
        print(f"{v:.2f} -> {q} -> {d:.4f}")


### Reflection

What makes collisions more likely, where two original values become the same integer code? Write one or two sentences before moving on.


<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Two values collide when they are closer together than one step between codes. The step is 1 / scale = (largest absolute value) / qmax. In our run it is 10.8 / 127, about 0.085, and 3.08 and 3.02 are only 0.06 apart, so both round to code 36.

So anything that makes the step wider makes collisions more likely:

- **Fewer bits**: fewer codes share the same range. In Part B, at 2 bits the nine values use only 3 codes.
- **A wider range**: one large value stretches the scale for every other value. Part D shows the extreme case: one outlier of 256 squeezes 24 ordinary values onto 3 codes.
- **Values that sit close together**, as many weights in a real model do.

</details>

## Part B - sweep over bit widths

The same vector is quantized with 8, 4, 3, and 2 bits. This shows why very low precision usually needs more careful methods than plain rounding.


### ✏️ Exercise 2 — Sweep the bit widths

**Goal:** see in three numbers per bit width how fewer bits mean fewer distinct codes and larger errors.

For each bit width, fill in three columns: how many distinct codes this vector
actually uses, and the mean and maximum absolute reconstruction error.

<details><summary><b>Hints</b> (click to open)</summary>

- `set(...)` keeps each distinct value once; `len` counts them. Turn `q` into a list first.
- The error of each value is `values - dq`. Take its absolute value first.

</details>

In [ ]:
# Exercise 2
rows = []
for bits in [8, 4, 3, 2]:
    q, s = absmax_quantize(values, bits=bits)
    dq = absmax_dequantize(q, s)
    rows.append({
        "bits": bits,
        "num_absmax_codes": 2 ** bits - 1,
        "num_codes_used_here": len(set(___)),
        "scale": s,
        "mean_abs_error": np.mean(np.abs(___ - ___)),
        "max_abs_error": np.___(np.abs(values - dq)),
    })

bit_sweep = pd.DataFrame(rows)
bit_sweep


In [ ]:
# Check for Exercise 2. Run this cell as is.
assert list(bit_sweep["bits"]) == [8, 4, 3, 2]
assert bit_sweep.loc[0, "num_codes_used_here"] == 8, "at 8 bits, 3.08 and 3.02 share code 36"
assert bit_sweep.loc[0, "mean_abs_error"] < bit_sweep.loc[3, "mean_abs_error"]
assert (bit_sweep["max_abs_error"] >= bit_sweep["mean_abs_error"]).all()
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
rows = []
for bits in [8, 4, 3, 2]:
    q, s = absmax_quantize(values, bits=bits)
    dq = absmax_dequantize(q, s)
    rows.append({
        "bits": bits,
        "num_absmax_codes": 2 ** bits - 1,
        "num_codes_used_here": len(set(q.tolist())),
        "scale": s,
        "mean_abs_error": np.mean(np.abs(values - dq)),
        "max_abs_error": np.max(np.abs(values - dq)),
    })

bit_sweep = pd.DataFrame(rows)
bit_sweep
```

</details>

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(bit_sweep["bits"], bit_sweep["mean_abs_error"], marker="o")
plt.gca().invert_xaxis()
plt.xlabel("bits")
plt.ylabel("mean absolute reconstruction error")
plt.title("Lower precision usually increases reconstruction error")
plt.show()


## Part C - asymmetric quantization for lopsided data

A quick reminder of what "activations" are: the intermediate output values produced as
data flows through a network's layers during a forward pass, not the weights. Weights are
fixed after training; activations are computed fresh from whatever input the model is
currently given, so their range can shift from one input to the next.

Here the values are all non-negative, like activations after a ReLU (ReLU zeroes out
anything negative, so everything downstream of it is $\geq 0$). Symmetric quantization
wastes half the code range on negative values that never occur.


### ✏️ Exercise 3 — Asymmetric quantization

**Goal:** use every code for values that are all 0 or more, by moving the range with a zero point.

Complete `asymmetric_quantize` and `asymmetric_dequantize`. The codes run from 0 to
2^bits − 1. The range [xmin, xmax] is mapped onto them, and the **zero point** is the code
that represents 0.0.

<details><summary><b>Hints</b> (click to open)</summary>

- Here the scale is the width of the range divided by the number of steps: the size of one step
  (a value per code). That is the inverse of absmax's scale (codes per unit value), so here you
  divide by it to quantize, where absmax multiplied.
- Quantize: divide by the scale, then shift by the zero point.
- Dequantize: undo the shift, then multiply by the scale.

</details>

In [ ]:
# Exercise 3
def asymmetric_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmin, qmax = 0, 2 ** bits - 1
    xmin, xmax = float(np.min(x)), float(np.max(x))
    if xmax == xmin:
        return np.zeros_like(x, dtype=np.int32), 1.0, 0
    scale = (___ - ___) / (qmax - qmin)
    zero_point = int(np.round(qmin - xmin / scale))
    q = np.round(x / scale + ___)
    q = np.clip(q, qmin, qmax).astype(np.int32)
    return q, scale, zero_point


def asymmetric_dequantize(q, scale, zero_point):
    return (np.asarray(q, dtype=np.float32) - ___) * ___

activations = np.array([0.0, 0.1, 0.4, 0.9, 1.0, 1.4, 2.0, 2.2, 2.8], dtype=np.float32)

q_sym, s_sym = absmax_quantize(activations, bits=4)
dq_sym = absmax_dequantize(q_sym, s_sym)

q_asym, s_asym, z_asym = asymmetric_quantize(activations, bits=4)
dq_asym = asymmetric_dequantize(q_asym, s_asym, z_asym)

pd.DataFrame({
    "activation": activations,
    "sym_code": q_sym,
    "sym_deq": np.round(dq_sym, 3),
    "asym_code": q_asym,
    "asym_deq": np.round(dq_asym, 3),
})


In [ ]:
# Check for Exercise 3. Run this cell as is.
assert q_asym.min() == 0 and q_asym.max() == 15, q_asym
assert z_asym == 0, "these activations start at 0.0, so code 0 represents 0.0"
_q, _s, _z = asymmetric_quantize(np.array([-1.0, 3.0]), bits=8)
assert (_q.tolist(), _z) == ([0, 255], 64), (_q.tolist(), _z)
assert np.mean(np.abs(activations - dq_asym)) < np.mean(np.abs(activations - dq_sym))
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def asymmetric_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmin, qmax = 0, 2 ** bits - 1
    xmin, xmax = float(np.min(x)), float(np.max(x))
    if xmax == xmin:
        return np.zeros_like(x, dtype=np.int32), 1.0, 0
    scale = (xmax - xmin) / (qmax - qmin)
    zero_point = int(np.round(qmin - xmin / scale))
    q = np.round(x / scale + zero_point)
    q = np.clip(q, qmin, qmax).astype(np.int32)
    return q, scale, zero_point


def asymmetric_dequantize(q, scale, zero_point):
    return (np.asarray(q, dtype=np.float32) - zero_point) * scale

activations = np.array([0.0, 0.1, 0.4, 0.9, 1.0, 1.4, 2.0, 2.2, 2.8], dtype=np.float32)

q_sym, s_sym = absmax_quantize(activations, bits=4)
dq_sym = absmax_dequantize(q_sym, s_sym)

q_asym, s_asym, z_asym = asymmetric_quantize(activations, bits=4)
dq_asym = asymmetric_dequantize(q_asym, s_asym, z_asym)

pd.DataFrame({
    "activation": activations,
    "sym_code": q_sym,
    "sym_deq": np.round(dq_sym, 3),
    "asym_code": q_asym,
    "asym_deq": np.round(dq_asym, 3),
})
```

</details>

In [ ]:
print("Symmetric mean abs error:", np.mean(np.abs(activations - dq_sym)))
print("Asymmetric mean abs error:", np.mean(np.abs(activations - dq_asym)))
print("Asymmetric scale:", s_asym, "zero point:", z_asym)


## Part D - outliers and clipping

Absmax quantization can waste most of the integer range on one extreme value. Clipping deliberately damages the outlier so the ordinary values are represented better.


In [ ]:
rng = np.random.default_rng(7)
ordinary = np.round(rng.uniform(-2, 2, size=24), 2)
with_outlier = np.concatenate([ordinary, [256.0]])

q_out, s_out = absmax_quantize(with_outlier, bits=8)
dq_out = absmax_dequantize(q_out, s_out)

ordinary_codes = q_out[:-1]
print("Distinct codes used by ordinary values, no clipping:", len(set(ordinary_codes.tolist())))
print("First 10 ordinary values/codes:")
print([(float(v), int(q)) for v, q in zip(with_outlier[:10], q_out[:10])])


### ✏️ Exercise 4 — Clip before quantizing

**Goal:** cap the range before quantizing, so that one outlier no longer sets the scale for every value.

Complete `quantize_with_clipping`: limit every value to the range
[−clip_range, clip_range] before absmax quantization.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.clip(x, low, high)` replaces values below `low` with `low` and above `high` with `high`.

</details>

In [ ]:
# Exercise 4
def quantize_with_clipping(x, clip_range=5.0, bits=8):
    clipped = np.clip(x, ___, ___)
    q, s = absmax_quantize(clipped, bits=bits)
    dq = absmax_dequantize(q, s)
    return clipped, q, dq, s

summary = []
for clip_range in [None, 10.0, 5.0, 2.5]:
    if clip_range is None:
        q, s = absmax_quantize(with_outlier, bits=8)
        dq = absmax_dequantize(q, s)
        label = "no clipping"
    else:
        _, q, dq, s = quantize_with_clipping(with_outlier, clip_range=clip_range, bits=8)
        label = f"clip [-{clip_range}, {clip_range}]"
    summary.append({
        "setting": label,
        "ordinary_codes_used": len(set(q[:-1].tolist())),
        "ordinary_MAE": np.mean(np.abs(with_outlier[:-1] - dq[:-1])),
        "all_values_MAE": np.mean(np.abs(with_outlier - dq)),
        "outlier_dequantized": dq[-1],
    })

pd.DataFrame(summary)


In [ ]:
# Check for Exercise 4. Run this cell as is.
_c, _q, _dq, _s = quantize_with_clipping(np.array([-10.0, 0.0, 1.0, 10.0]), clip_range=2.0)
assert _c.tolist() == [-2, 0, 1, 2], _c.tolist()
assert _q.tolist() == [-127, 0, 64, 127], _q.tolist()
_df = pd.DataFrame(summary)
assert _df["ordinary_codes_used"].iloc[-1] > _df["ordinary_codes_used"].iloc[0]
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def quantize_with_clipping(x, clip_range=5.0, bits=8):
    clipped = np.clip(x, -clip_range, clip_range)
    q, s = absmax_quantize(clipped, bits=bits)
    dq = absmax_dequantize(q, s)
    return clipped, q, dq, s

summary = []
for clip_range in [None, 10.0, 5.0, 2.5]:
    if clip_range is None:
        q, s = absmax_quantize(with_outlier, bits=8)
        dq = absmax_dequantize(q, s)
        label = "no clipping"
    else:
        _, q, dq, s = quantize_with_clipping(with_outlier, clip_range=clip_range, bits=8)
        label = f"clip [-{clip_range}, {clip_range}]"
    summary.append({
        "setting": label,
        "ordinary_codes_used": len(set(q[:-1].tolist())),
        "ordinary_MAE": np.mean(np.abs(with_outlier[:-1] - dq[:-1])),
        "all_values_MAE": np.mean(np.abs(with_outlier - dq)),
        "outlier_dequantized": dq[-1],
    })

pd.DataFrame(summary)
```

</details>

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(np.arange(len(with_outlier)), with_outlier, label="original")
_, _, dq_clip5, _ = quantize_with_clipping(with_outlier, clip_range=5.0, bits=8)
plt.scatter(np.arange(len(dq_clip5)), dq_clip5, marker="x", label="after clip+quantize+dequantize")
plt.xlabel("index")
plt.ylabel("value")
plt.title("Clipping preserves ordinary values better but damages the outlier")
plt.legend()
plt.show()


### Reflection

Was clipping worth it here? Answer twice: once if the ordinary values matter most, and once if the outlier matters just as much as every other value.


<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

**If the ordinary values matter most:** yes, clearly. In our run, with no clipping the 24 ordinary values share only 3 codes, and their average error is about 0.5, which is large for values between −2 and 2. Clipping at ±2.5 gives them 23 different codes and makes their error about a hundred times smaller. Every clipping range we tried helped, and the tighter ones helped most.

**If the outlier matters just as much as every other value:** no. Clipping turns 256 into 10, 5 or 2.5, an error of about 250 on that one value. Averaged over all 25 values, that single error raises the error from about 0.5 to about 10, twenty times worse than not clipping at all.

So clipping is a bet that the outlier is less important than the rest. Another way out is to quantize small groups of values separately, each group with its own scale: then the outlier only widens the step for its own group. The advanced version builds this in Exercise 5.

</details>

## What you learned: check yourself

Answer each question from what you ran and read above. If one is hard, go back to the
step it comes from.

1. In absmax quantization, which value sets the scale? When do two different values end up with the same code?
2. From 8 bits down to 2, how did the number of distinct codes and the mean error change?
3. For values that are all 0 or more, which gave the smaller error, symmetric or asymmetric quantization? Why?
4. With the outlier and no clipping, why do the ordinary values share only a few codes? When would you not clip?

## Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow. At minimum, comment on:

1. how the scale is computed;
2. why rounding creates quantization error;
3. why clipping helps ordinary values in the outlier example.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.
